# fNIRS example
Reuse one baseline phase to process two fNIRS task phases independently.

In [ ]:
from pathlib import Path
import numpy as np
from multimodalphysiokit.io import read_biosignalsplux_hdf5
from multimodalphysiokit.processors import FNIRSProcessor
from multimodalphysiokit.utils import plot_signal_with_phases

data_file = Path("examples/sample_data/synthetic_multimodal_180s.hdf5")
recording = read_biosignalsplux_hdf5(data_file, fnirs_column_order=(0, 1))
recording.signal_names()

In [ ]:
phase_intervals_seconds = np.array(
    [[0.0, 60.0, 120.0], [60.0, 120.0, 180.0]]
)
recording.set_phases(
    phase_intervals_seconds,
    ["baseline", "task_1", "task_2"],
)
plot_signal_with_phases(recording, "fnirs_red", title="fNIRS red current", show=True)
plot_signal_with_phases(recording, "fnirs_infrared", title="fNIRS infrared current", show=True)

In [ ]:
processor = FNIRSProcessor(age_years=25)
baseline_red_value, baseline_infrared_value = processor.compute_baseline_values(
    recording.extract_phase("fnirs_red", "baseline"),
    recording.extract_phase("fnirs_infrared", "baseline"),
)

In [ ]:
fnirs_results = recording.process_phases(
    processor,
    ("fnirs_red", "fnirs_infrared"),
    phases=("task_1", "task_2"),
    process_kwargs={
        "baseline_red_value": baseline_red_value,
        "baseline_infrared_value": baseline_infrared_value,
    },
)

In [ ]:
{
    phase: {
        "feature_count": len(result.features),
        "fnirs_oxy_mean": result.features["fnirs_oxy_mean"],
        "fnirs_deoxy_mean": result.features["fnirs_deoxy_mean"],
    }
    for phase, result in fnirs_results.items()
}